# Analisis de estrategias SMC — Top 10

Este notebook carga los resultados del grid search de 100 estrategias y los visualiza.

> **Prerequisito:** ejecutar primero `python -m backtests.strategy_selector` para generar `data/results.csv`

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Ruta al CSV (relativa al notebook)
CSV_PATH = os.path.join("..", "data", "results.csv")

df = pd.read_csv(CSV_PATH)

# Filtrar estrategias con trades suficientes
df_valid = df[df["trades"] >= 3].copy()
df_top10 = df_valid.head(10).copy()

print(f"Estrategias totales: {len(df)}")
print(f"Con >= 3 trades:     {len(df_valid)}")
df_top10

## 1 · Top 10 por PnL total

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

labels = [
    f"sw={r.swing_window} hold={r.max_hold}\nfvg={'si' if r.require_fvg else 'no'} choch={'si' if r.use_choch_filter else 'no'}"
    for _, r in df_top10.iterrows()
]
colors = ["steelblue" if v >= 0 else "tomato" for v in df_top10["total_pnl"]]

bars = ax.bar(range(len(df_top10)), df_top10["total_pnl"] * 100, color=colors)
ax.set_xticks(range(len(df_top10)))
ax.set_xticklabels(labels, fontsize=8)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.2f%%"))
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Top 10 estrategias — PnL total (%)", fontsize=13)
ax.set_ylabel("PnL (%)")
ax.set_xlabel("Parametros")

for bar, val in zip(bars, df_top10["total_pnl"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + (0.001 if val >= 0 else -0.003),
        f"{val*100:+.2f}%",
        ha="center", va="bottom", fontsize=8
    )

plt.tight_layout()
plt.show()

## 2 · PnL vs Winrate (todas las estrategias validas)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sc = ax.scatter(
    df_valid["winrate"] * 100,
    df_valid["total_pnl"] * 100,
    c=df_valid["trades"],
    cmap="viridis",
    s=60,
    alpha=0.75,
    edgecolors="none",
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("N° de trades")

# Marcar top 10
ax.scatter(
    df_top10["winrate"] * 100,
    df_top10["total_pnl"] * 100,
    marker="*", s=200, color="red", label="Top 10", zorder=5
)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.axvline(50, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel("Winrate (%)")
ax.set_ylabel("PnL total (%)")
ax.set_title("PnL vs Winrate — todas las estrategias validas", fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

## 3 · Heatmap: swing_window vs max_hold (PnL medio)

In [ ]:
pivot = df_valid.pivot_table(
    values="total_pnl",
    index="swing_window",
    columns="max_hold",
    aggfunc="mean"
) * 100

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto")

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("max_hold (velas)")
ax.set_ylabel("swing_window (velas)")
ax.set_title("PnL medio (%) por swing_window × max_hold", fontsize=13)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(j, i, f"{val:.2f}%", ha="center", va="center", fontsize=8,
                color="black")

plt.colorbar(im, ax=ax, label="PnL medio (%)")
plt.tight_layout()
plt.show()

## 4 · Efecto de require_fvg y use_choch_filter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, col, title, labels in zip(
    axes,
    ["require_fvg", "use_choch_filter"],
    ["Efecto de require_fvg", "Efecto de use_choch_filter"],
    [["Sin FVG", "Con FVG"], ["Sin filtro CHoCH", "Con filtro CHoCH"]],
):
    means = df_valid.groupby(col)["total_pnl"].mean() * 100
    colors = ["steelblue" if v >= 0 else "tomato" for v in means]
    bars = ax.bar(labels, means.values, color=colors)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.2f%%"))
    ax.set_title(title, fontsize=11)
    ax.set_ylabel("PnL medio (%)")
    for bar, val in zip(bars, means):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (0.0005 if val >= 0 else -0.002),
            f"{val:+.3f}%",
            ha="center", va="bottom", fontsize=10
        )

plt.suptitle("Impacto de los filtros booleanos en PnL medio", fontsize=13)
plt.tight_layout()
plt.show()